# SIGMA Phase 0: Jamba-1.5-Mini Baseline Benchmarks

**Project**: SIGMA (State-space Integrated Mixture-of-experts for Generative Agents)
**Author**: Shubham
**Course**: AAI/CPE/EE 800 - Spring 2026
**Date**: February 2026

---

## Objective

Establish baseline performance of **vanilla Jamba-1.5-Mini** on four task-specific benchmarks:

| Task Type | Benchmark | Metric | Expected Baseline |
|-----------|-----------|--------|-------------------|
| Reasoning | GSM8K (1319 test) | Accuracy | ~45-55% |
| Tool-use | ToolBench (G1-Instruction) | Pass Rate | ~40-50% |
| Retrieval | HotpotQA | F1/EM | ~35-45% |
| General | MMLU (subset) | Accuracy | ~55-65% |

## Why This Matters

These baselines will be compared against **SIGMA's task-aware routing** to demonstrate:
1. Whether task-specialized LoRA adapters improve over vanilla
2. Whether SSM-based routing outperforms random routing
3. The magnitude of improvement from intelligent routing

---

## Hardware Requirements

- **GPU**: A100 (40/80GB) or H100 recommended
- **RAM**: 32GB+ system RAM
- **Storage**: ~50GB for model + datasets
- **Estimated Runtime**: 4-8 hours total

---

## Table of Contents

1. [Environment Setup](#1-environment-setup)
2. [Model Loading](#2-model-loading)
3. [Benchmark 1: GSM8K (Reasoning)](#3-benchmark-1-gsm8k-reasoning)
4. [Benchmark 2: ToolBench (Tool-use)](#4-benchmark-2-toolbench-tool-use)
5. [Benchmark 3: HotpotQA (Retrieval)](#5-benchmark-3-hotpotqa-retrieval)
6. [Benchmark 4: MMLU (General)](#6-benchmark-4-mmlu-general)
7. [Results Summary](#7-results-summary)
8. [Export Results](#8-export-results)

---

## 1. Environment Setup

In [1]:
# Cell 1: Install core dependencies
!pip install -q transformers>=4.39.0 accelerate>=0.27.0
!pip install -q datasets evaluate sacrebleu rouge_score
!pip install -q bitsandbytes scipy sentencepiece
!pip install -q tqdm pandas matplotlib seaborn

ERROR: huggingface-hub 0.36.2 has requirement packaging>=20.9, but you'll have packaging 20.3 which is incompatible.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
ERROR: huggingface-hub 0.36.2 has requirement packaging>=20.9, but you'

In [2]:
# Cell 2: Install lm-evaluation-harness for standardized benchmarks
!pip install -q lm-eval>=0.4.0

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.


In [3]:
# Cell 3: Install Mamba/Jamba specific dependencies
# Note: These may require compilation - takes ~10-15 minutes
!pip install -q mamba-ssm>=1.2.0 causal-conv1d>=1.2.0 --no-build-isolation

    ERROR: Command errored out with exit status 1:
     command: /usr/bin/python3 -c 'import sys, setuptools, tokenize; sys.argv[0] = '"'"'/tmp/pip-install-cff43fj3/mamba-ssm/setup.py'"'"'; __file__='"'"'/tmp/pip-install-cff43fj3/mamba-ssm/setup.py'"'"';f=getattr(tokenize, '"'"'open'"'"', open)(__file__);code=f.read().replace('"'"'\r\n'"'"', '"'"'\n'"'"');f.close();exec(compile(code, __file__, '"'"'exec'"'"'))' egg_info --egg-base /tmp/pip-install-cff43fj3/mamba-ssm/pip-egg-info
         cwd: /tmp/pip-install-cff43fj3/mamba-ssm/
    Complete output (10 lines):
    Traceback (most recent call last):
      File "<string>", line 1, in <module>
      File "/tmp/pip-install-cff43fj3/mamba-ssm/setup.py", line 171, in <module>
        raise RuntimeError(
    RuntimeError: mamba_ssm is only supported on CUDA 11.6 and above.  Note: make sure nvcc has a supported version by running nvcc -V.
    
    
    torch.__version__  = 2.4.1+cu121
    
    
    ----------------------------------------
ERRO

In [4]:
# Cell 4: Core imports
import os
import json
import time
import torch
import numpy as np
import pandas as pd
from datetime import datetime
from tqdm.auto import tqdm
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Set random seeds for reproducibility
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

PyTorch version: 2.2.2+cu121
CUDA available: True
GPU: NVIDIA RTX A6000
GPU Memory: 51.0 GB


In [5]:
# Cell 5: Configuration
CONFIG = {
    # Model settings
    "model_name": "ai21labs/AI21-Jamba-1.5-Mini",
    "torch_dtype": "float16",
    "device_map": "auto",

    # Generation settings
    "max_new_tokens": 512,
    "temperature": 0.0,  # Greedy decoding for reproducibility
    "do_sample": False,

    # Benchmark settings
    "gsm8k_num_fewshot": 8,
    "mmlu_num_fewshot": 5,
    "batch_size": 1,  # Jamba is memory-hungry, start conservative

    # Evaluation limits (set to None for full evaluation)
    "gsm8k_limit": 100,  # Full: 1319 test samples
    "toolbench_limit": 200,  # Subset for efficiency
    "hotpotqa_limit": 500,  # Subset for efficiency
    "mmlu_limit": 200,  # Full MMLU

    # Output paths
    "results_dir": "./sigma_baseline_results",
    "timestamp": datetime.now().strftime("%Y%m%d_%H%M%S")
}

# Create results directory
os.makedirs(CONFIG["results_dir"], exist_ok=True)
print(f"Results will be saved to: {CONFIG['results_dir']}")

Results will be saved to: ./sigma_baseline_results


In [6]:
from huggingface_hub import login
login(token="YOUR_HUGGINGFACE_TOKEN_HERE")

In [7]:
torch.cuda.empty_cache()

---

## 2. Model Loading

In [8]:
# Cell 6: Load Jamba-1.5-Mini with FIXED configuration
from transformers import AutoModelForCausalLM, AutoTokenizer, AutoConfig

print(f"Loading {CONFIG['model_name']}...")
print("This may take 5-10 minutes for first download (~50GB total parameters)")
print("\n⚠️ Using compatibility mode (no fast Mamba kernels)")

start_time = time.time()

# Step 1: Load tokenizer
print("\n[1/3] Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(
    CONFIG["model_name"],
    trust_remote_code=True
)

# Ensure padding token is set
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.pad_token_id = tokenizer.eos_token_id

print(f"   Tokenizer loaded. Vocab size: {tokenizer.vocab_size}")

# Step 2: Load and modify config
print("\n[2/3] Loading model configuration...")
config = AutoConfig.from_pretrained(
    CONFIG["model_name"],
    trust_remote_code=True
)

# CRITICAL: Disable Mamba kernels to avoid CUDA issues
if hasattr(config, 'use_mamba_kernels'):
    config.use_mamba_kernels = False
    print("   ⚙️ Mamba kernels: DISABLED (fallback mode)")

# Step 3: Load model with proper settings
print("\n[3/3] Loading model weights...")
print("   This takes ~5-8 minutes on A100...")

model = AutoModelForCausalLM.from_pretrained(
    CONFIG["model_name"],
    config=config,
    torch_dtype=getattr(torch, CONFIG["torch_dtype"]),
    device_map="auto",
    trust_remote_code=True,
    # CRITICAL FIX: These settings prevent meta device issues
    low_cpu_mem_usage=False,  # Avoid meta tensors
    offload_buffers=False,    # Keep everything on GPU
)

model.eval()

load_time = time.time() - start_time
print(f"\n" + "="*50)
print(f"✅ MODEL LOADED SUCCESSFULLY")
print(f"="*50)
print(f"Time: {load_time:.1f} seconds ({load_time/60:.1f} minutes)")
print(f"Parameters: {sum(p.numel() for p in model.parameters()) / 1e9:.2f}B total")
print(f"GPU Memory: {torch.cuda.memory_allocated() / 1e9:.2f} GB allocated")
print(f"GPU Memory: {torch.cuda.max_memory_allocated() / 1e9:.2f} GB peak")

You are using the default legacy behaviour of the <class 'transformers.models.llama.tokenization_llama_fast.LlamaTokenizerFast'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565 - if you loaded a llama tokenizer from a GGUF file you can ignore this message.


Loading ai21labs/AI21-Jamba-1.5-Mini...
This may take 5-10 minutes for first download (~50GB total parameters)

⚠️ Using compatibility mode (no fast Mamba kernels)

[1/3] Loading tokenizer...
   Tokenizer loaded. Vocab size: 65536

[2/3] Loading model configuration...
   ⚙️ Mamba kernels: DISABLED (fallback mode)

[3/3] Loading model weights...
   This takes ~5-8 minutes on A100...


ValueError: Passing along a `device_map` requires `low_cpu_mem_usage=True`

In [ ]:
# Cell 7: Quick sanity check - verify model works
test_prompt = "What is 2 + 2? Answer:"
inputs = tokenizer(test_prompt, return_tensors="pt").to(model.device)

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=20,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id
    )

response = tokenizer.decode(outputs[0], skip_special_tokens=True)
print("Sanity Check:")
print(f"Prompt: {test_prompt}")
print(f"Response: {response}")
print("\n✅ Model is working correctly!")

---

## 3. Benchmark 1: GSM8K (Reasoning)

**Description**: Grade school math word problems requiring multi-step reasoning.

**Metric**: Exact match accuracy on numerical answers.

**Expected Baseline**: ~45-55% with 8-shot prompting

In [ ]:
# Cell 8: Load GSM8K dataset
from datasets import load_dataset

print("Loading GSM8K dataset...")
gsm8k = load_dataset("gsm8k", "main")
print(f"Train samples: {len(gsm8k['train'])}")
print(f"Test samples: {len(gsm8k['test'])}")

# Show example
print("\n--- Example Problem ---")
print(f"Question: {gsm8k['test'][0]['question']}")
print(f"\nAnswer: {gsm8k['test'][0]['answer']}")

In [ ]:
# Cell 9: GSM8K evaluation functions
import re

def extract_gsm8k_answer(text):
    """Extract numerical answer from GSM8K format (#### NUMBER)"""
    # Look for #### pattern
    match = re.search(r'####\s*(-?[\d,]+\.?\d*)', text)
    if match:
        return match.group(1).replace(',', '')

    # Fallback: find last number in text
    numbers = re.findall(r'-?[\d,]+\.?\d*', text)
    if numbers:
        return numbers[-1].replace(',', '')
    return None

def create_gsm8k_prompt(question, fewshot_examples=None):
    """Create few-shot prompt for GSM8K"""
    prompt = """Solve the following math problem step by step. Show your work and put your final numerical answer after ####.

"""

    if fewshot_examples:
        for ex in fewshot_examples:
            prompt += f"Question: {ex['question']}\n"
            prompt += f"Answer: {ex['answer']}\n\n"

    prompt += f"Question: {question}\nAnswer:"
    return prompt

# Test extraction
test_answer = "The answer is #### 42"
print(f"Extraction test: '{test_answer}' -> {extract_gsm8k_answer(test_answer)}")

In [ ]:
# Cell 10: Run GSM8K evaluation
def evaluate_gsm8k(model, tokenizer, dataset, num_fewshot=8, limit=None):
    """Evaluate model on GSM8K benchmark"""

    # Get few-shot examples from training set
    fewshot_examples = list(dataset['train'].select(range(num_fewshot)))

    # Prepare test set
    test_data = dataset['test']
    if limit:
        test_data = test_data.select(range(min(limit, len(test_data))))

    correct = 0
    total = 0
    results = []

    print(f"\nEvaluating GSM8K ({len(test_data)} samples)...")

    for item in tqdm(test_data, desc="GSM8K"):
        prompt = create_gsm8k_prompt(item['question'], fewshot_examples)

        inputs = tokenizer(
            prompt,
            return_tensors="pt",
            truncation=True,
            max_length=2048
        ).to(model.device)

        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=512,
                do_sample=False,
                pad_token_id=tokenizer.eos_token_id,
                eos_token_id=tokenizer.eos_token_id
            )

        response = tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)

        # Extract answers
        predicted = extract_gsm8k_answer(response)
        expected = extract_gsm8k_answer(item['answer'])

        # Compare
        is_correct = False
        if predicted and expected:
            try:
                is_correct = abs(float(predicted) - float(expected)) < 1e-6
            except ValueError:
                is_correct = predicted == expected

        if is_correct:
            correct += 1
        total += 1

        results.append({
            'question': item['question'],
            'expected': expected,
            'predicted': predicted,
            'correct': is_correct,
            'response': response[:500]  # Truncate for storage
        })

        # Clear GPU cache periodically
        if total % 50 == 0:
            torch.cuda.empty_cache()

    accuracy = correct / total
    return {
        'accuracy': accuracy,
        'correct': correct,
        'total': total,
        'results': results
    }

print("GSM8K evaluation function ready.")

In [ ]:
# Cell 11: Execute GSM8K evaluation
print("="*60)
print("BENCHMARK 1: GSM8K (REASONING)")
print("="*60)

gsm8k_start = time.time()
gsm8k_results = evaluate_gsm8k(
    model,
    tokenizer,
    gsm8k,
    num_fewshot=CONFIG["gsm8k_num_fewshot"],
    limit=CONFIG["gsm8k_limit"]
)
gsm8k_time = time.time() - gsm8k_start

print(f"\n{'='*40}")
print(f"GSM8K RESULTS")
print(f"{'='*40}")
print(f"Accuracy: {gsm8k_results['accuracy']:.2%}")
print(f"Correct: {gsm8k_results['correct']}/{gsm8k_results['total']}")
print(f"Time: {gsm8k_time/60:.1f} minutes")

# Save results
gsm8k_output = {
    'benchmark': 'gsm8k',
    'model': CONFIG['model_name'],
    'accuracy': gsm8k_results['accuracy'],
    'correct': gsm8k_results['correct'],
    'total': gsm8k_results['total'],
    'num_fewshot': CONFIG['gsm8k_num_fewshot'],
    'runtime_seconds': gsm8k_time,
    'timestamp': CONFIG['timestamp']
}

with open(f"{CONFIG['results_dir']}/gsm8k_results.json", 'w') as f:
    json.dump(gsm8k_output, f, indent=2)
print(f"\n✅ Results saved to {CONFIG['results_dir']}/gsm8k_results.json")

---

## 4. Benchmark 2: ToolBench (Tool-use)

**Description**: Tests model's ability to correctly use tools/APIs in response to queries.

**Metric**: Pass rate - correct tool selection and argument formatting.

**Expected Baseline**: ~40-50%

In [ ]:
# Cell 12: Define tool-use evaluation dataset and prompts
# Using a simplified ToolBench-style evaluation

TOOL_DEFINITIONS = {
    "calculator": {
        "name": "calculator",
        "description": "Performs mathematical calculations",
        "parameters": {
            "expression": "string - mathematical expression to evaluate"
        }
    },
    "web_search": {
        "name": "web_search",
        "description": "Searches the web for information",
        "parameters": {
            "query": "string - search query"
        }
    },
    "weather": {
        "name": "weather",
        "description": "Gets current weather for a location",
        "parameters": {
            "location": "string - city name or coordinates"
        }
    },
    "calendar": {
        "name": "calendar",
        "description": "Manages calendar events",
        "parameters": {
            "action": "string - create/read/update/delete",
            "event": "string - event details"
        }
    },
    "email": {
        "name": "email",
        "description": "Sends or reads emails",
        "parameters": {
            "action": "string - send/read",
            "to": "string - recipient email (for send)",
            "subject": "string - email subject",
            "body": "string - email body"
        }
    }
}

TOOL_TEST_CASES = [
    # Calculator tasks
    {"query": "What is 15% of 240?", "expected_tool": "calculator", "expected_args": ["expression"]},
    {"query": "Calculate the square root of 144", "expected_tool": "calculator", "expected_args": ["expression"]},
    {"query": "What is 3.14159 * 10^2?", "expected_tool": "calculator", "expected_args": ["expression"]},
    {"query": "Divide 1000 by 7 and round to 2 decimal places", "expected_tool": "calculator", "expected_args": ["expression"]},

    # Web search tasks
    {"query": "Find information about the latest Mars rover mission", "expected_tool": "web_search", "expected_args": ["query"]},
    {"query": "Search for recipes using chicken and broccoli", "expected_tool": "web_search", "expected_args": ["query"]},
    {"query": "Look up the population of Tokyo", "expected_tool": "web_search", "expected_args": ["query"]},
    {"query": "Who won the Nobel Prize in Physics last year?", "expected_tool": "web_search", "expected_args": ["query"]},

    # Weather tasks
    {"query": "What's the weather like in New York City?", "expected_tool": "weather", "expected_args": ["location"]},
    {"query": "Will it rain in London tomorrow?", "expected_tool": "weather", "expected_args": ["location"]},
    {"query": "Check the temperature in Paris", "expected_tool": "weather", "expected_args": ["location"]},
    {"query": "Is it snowing in Denver right now?", "expected_tool": "weather", "expected_args": ["location"]},

    # Calendar tasks
    {"query": "Schedule a meeting for tomorrow at 3pm", "expected_tool": "calendar", "expected_args": ["action", "event"]},
    {"query": "What events do I have next week?", "expected_tool": "calendar", "expected_args": ["action"]},
    {"query": "Create a reminder for my dentist appointment on Friday", "expected_tool": "calendar", "expected_args": ["action", "event"]},
    {"query": "Cancel my meeting with John", "expected_tool": "calendar", "expected_args": ["action", "event"]},

    # Email tasks
    {"query": "Send an email to bob@example.com about the project update", "expected_tool": "email", "expected_args": ["action", "to", "subject"]},
    {"query": "Check my unread emails", "expected_tool": "email", "expected_args": ["action"]},
    {"query": "Reply to Sarah's email about the meeting", "expected_tool": "email", "expected_args": ["action", "to"]},
    {"query": "Draft an email to the team about the deadline extension", "expected_tool": "email", "expected_args": ["action", "subject", "body"]},
]

# Expand test cases by generating variations
EXPANDED_TOOL_TESTS = TOOL_TEST_CASES * 10  # 200 total test cases

print(f"Tool-use test cases: {len(EXPANDED_TOOL_TESTS)}")
print(f"Tools defined: {list(TOOL_DEFINITIONS.keys())}")

In [ ]:
# Cell 13: Tool-use evaluation functions
def create_tool_prompt(query, tools):
    """Create prompt for tool selection task"""
    tools_desc = "\n".join([
        f"- {t['name']}: {t['description']}. Parameters: {t['parameters']}"
        for t in tools.values()
    ])

    prompt = f"""You are an assistant that can use tools to help users. Given a user query, select the appropriate tool and provide the necessary arguments.

Available tools:
{tools_desc}

Respond in JSON format with "tool" and "arguments" fields.

Example:
User: What is 5 + 3?
Assistant: {{"tool": "calculator", "arguments": {{"expression": "5 + 3"}}}}

User: {query}
Assistant:"""
    return prompt

def parse_tool_response(response):
    """Parse model response to extract tool and arguments"""
    try:
        # Try to find JSON in response
        import re
        json_match = re.search(r'\{[^{}]*\}', response)
        if json_match:
            parsed = json.loads(json_match.group())
            return parsed.get('tool'), parsed.get('arguments', {})
    except json.JSONDecodeError:
        pass

    # Fallback: look for tool names in response
    response_lower = response.lower()
    for tool_name in TOOL_DEFINITIONS.keys():
        if tool_name in response_lower:
            return tool_name, {}

    return None, {}

def evaluate_tool_response(predicted_tool, predicted_args, expected_tool, expected_args):
    """Evaluate if tool selection and arguments are correct"""
    # Check tool selection
    tool_correct = predicted_tool == expected_tool

    # Check if required arguments are present
    args_correct = True
    if predicted_args and expected_args:
        for arg in expected_args:
            if arg not in predicted_args:
                args_correct = False
                break

    return tool_correct, args_correct, tool_correct and args_correct

print("Tool-use evaluation functions ready.")

In [ ]:
# Cell 14: Run ToolBench-style evaluation
def evaluate_toolbench(model, tokenizer, test_cases, tools, limit=None):
    """Evaluate model on tool-use benchmark"""

    if limit:
        test_cases = test_cases[:limit]

    results = []
    tool_correct = 0
    args_correct = 0
    full_correct = 0
    total = 0

    print(f"\nEvaluating Tool-use ({len(test_cases)} samples)...")

    for case in tqdm(test_cases, desc="ToolBench"):
        prompt = create_tool_prompt(case['query'], tools)

        inputs = tokenizer(
            prompt,
            return_tensors="pt",
            truncation=True,
            max_length=2048
        ).to(model.device)

        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=150,
                do_sample=False,
                pad_token_id=tokenizer.eos_token_id
            )

        response = tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)

        pred_tool, pred_args = parse_tool_response(response)
        is_tool_correct, is_args_correct, is_full_correct = evaluate_tool_response(
            pred_tool, pred_args,
            case['expected_tool'], case['expected_args']
        )

        if is_tool_correct:
            tool_correct += 1
        if is_args_correct:
            args_correct += 1
        if is_full_correct:
            full_correct += 1
        total += 1

        results.append({
            'query': case['query'],
            'expected_tool': case['expected_tool'],
            'predicted_tool': pred_tool,
            'expected_args': case['expected_args'],
            'predicted_args': pred_args,
            'tool_correct': is_tool_correct,
            'full_correct': is_full_correct,
            'response': response[:300]
        })

        if total % 50 == 0:
            torch.cuda.empty_cache()

    return {
        'tool_accuracy': tool_correct / total,
        'args_accuracy': args_correct / total,
        'full_accuracy': full_correct / total,
        'tool_correct': tool_correct,
        'args_correct': args_correct,
        'full_correct': full_correct,
        'total': total,
        'results': results
    }

print("ToolBench evaluation function ready.")

In [ ]:
# Cell 15: Execute ToolBench evaluation
print("="*60)
print("BENCHMARK 2: TOOLBENCH (TOOL-USE)")
print("="*60)

toolbench_start = time.time()
toolbench_results = evaluate_toolbench(
    model,
    tokenizer,
    EXPANDED_TOOL_TESTS,
    TOOL_DEFINITIONS,
    limit=CONFIG["toolbench_limit"]
)
toolbench_time = time.time() - toolbench_start

print(f"\n{'='*40}")
print(f"TOOLBENCH RESULTS")
print(f"{'='*40}")
print(f"Tool Selection Accuracy: {toolbench_results['tool_accuracy']:.2%}")
print(f"Arguments Accuracy: {toolbench_results['args_accuracy']:.2%}")
print(f"Full Pass Rate: {toolbench_results['full_accuracy']:.2%}")
print(f"Correct: {toolbench_results['full_correct']}/{toolbench_results['total']}")
print(f"Time: {toolbench_time/60:.1f} minutes")

# Save results
toolbench_output = {
    'benchmark': 'toolbench',
    'model': CONFIG['model_name'],
    'tool_accuracy': toolbench_results['tool_accuracy'],
    'args_accuracy': toolbench_results['args_accuracy'],
    'full_accuracy': toolbench_results['full_accuracy'],
    'tool_correct': toolbench_results['tool_correct'],
    'full_correct': toolbench_results['full_correct'],
    'total': toolbench_results['total'],
    'runtime_seconds': toolbench_time,
    'timestamp': CONFIG['timestamp']
}

with open(f"{CONFIG['results_dir']}/toolbench_results.json", 'w') as f:
    json.dump(toolbench_output, f, indent=2)
print(f"\n✅ Results saved to {CONFIG['results_dir']}/toolbench_results.json")

---

## 5. Benchmark 3: HotpotQA (Retrieval/Multi-hop Reasoning)

**Description**: Multi-hop question answering requiring information from multiple passages.

**Metric**: F1 and Exact Match (EM) scores.

**Expected Baseline**: ~35-45% F1

In [ ]:
# Cell 16: Load HotpotQA dataset
print("Loading HotpotQA dataset...")
hotpotqa = load_dataset("hotpot_qa", "fullwiki")
print(f"Train samples: {len(hotpotqa['train'])}")
print(f"Validation samples: {len(hotpotqa['validation'])}")

# Show example
example = hotpotqa['validation'][0]
print("\n--- Example ---")
print(f"Question: {example['question']}")
print(f"Answer: {example['answer']}")
print(f"Type: {example['type']}")
print(f"Level: {example['level']}")

In [ ]:
# Cell 17: HotpotQA evaluation functions
import string
from collections import Counter

def normalize_answer(s):
    """Normalize answer for comparison"""
    def remove_articles(text):
        return re.sub(r'\b(a|an|the)\b', ' ', text)
    def white_space_fix(text):
        return ' '.join(text.split())
    def remove_punc(text):
        exclude = set(string.punctuation)
        return ''.join(ch for ch in text if ch not in exclude)
    def lower(text):
        return text.lower()
    return white_space_fix(remove_articles(remove_punc(lower(s))))

def f1_score(prediction, ground_truth):
    """Compute F1 score between prediction and ground truth"""
    pred_tokens = normalize_answer(prediction).split()
    truth_tokens = normalize_answer(ground_truth).split()

    if len(pred_tokens) == 0 or len(truth_tokens) == 0:
        return int(pred_tokens == truth_tokens)

    common = Counter(pred_tokens) & Counter(truth_tokens)
    num_same = sum(common.values())

    if num_same == 0:
        return 0

    precision = num_same / len(pred_tokens)
    recall = num_same / len(truth_tokens)
    f1 = (2 * precision * recall) / (precision + recall)
    return f1

def exact_match(prediction, ground_truth):
    """Compute exact match score"""
    return int(normalize_answer(prediction) == normalize_answer(ground_truth))

def create_hotpotqa_prompt(question, context=None):
    """Create prompt for HotpotQA"""
    prompt = """Answer the following question. Give a short, direct answer.

"""
    if context:
        prompt += f"Context: {context}\n\n"

    prompt += f"Question: {question}\nAnswer:"
    return prompt

# Test F1 calculation
test_pred = "The answer is Paris"
test_truth = "Paris"
print(f"F1 test: pred='{test_pred}', truth='{test_truth}' -> F1={f1_score(test_pred, test_truth):.3f}")

In [ ]:
# Cell 18: Run HotpotQA evaluation
def evaluate_hotpotqa(model, tokenizer, dataset, limit=None):
    """Evaluate model on HotpotQA benchmark"""

    val_data = dataset['validation']
    if limit:
        val_data = val_data.select(range(min(limit, len(val_data))))

    results = []
    total_f1 = 0
    total_em = 0
    total = 0

    print(f"\nEvaluating HotpotQA ({len(val_data)} samples)...")

    for item in tqdm(val_data, desc="HotpotQA"):
        # Build context from supporting facts
        context_parts = []
        for title, sentences in zip(item['context']['title'], item['context']['sentences']):
            context_parts.append(f"{title}: {' '.join(sentences)}")
        context = ' '.join(context_parts[:3])  # Limit context length

        prompt = create_hotpotqa_prompt(item['question'], context[:1500])

        inputs = tokenizer(
            prompt,
            return_tensors="pt",
            truncation=True,
            max_length=2048
        ).to(model.device)

        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=100,
                do_sample=False,
                pad_token_id=tokenizer.eos_token_id
            )

        response = tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)

        # Extract first line/sentence as answer
        predicted_answer = response.split('\n')[0].strip()
        if '.' in predicted_answer:
            predicted_answer = predicted_answer.split('.')[0]

        # Compute metrics
        f1 = f1_score(predicted_answer, item['answer'])
        em = exact_match(predicted_answer, item['answer'])

        total_f1 += f1
        total_em += em
        total += 1

        results.append({
            'question': item['question'],
            'expected': item['answer'],
            'predicted': predicted_answer,
            'f1': f1,
            'em': em,
            'type': item['type'],
            'level': item['level']
        })

        if total % 50 == 0:
            torch.cuda.empty_cache()

    return {
        'f1': total_f1 / total,
        'em': total_em / total,
        'total': total,
        'results': results
    }

print("HotpotQA evaluation function ready.")

In [ ]:
# Cell 19: Execute HotpotQA evaluation
print("="*60)
print("BENCHMARK 3: HOTPOTQA (RETRIEVAL/MULTI-HOP)")
print("="*60)

hotpotqa_start = time.time()
hotpotqa_results = evaluate_hotpotqa(
    model,
    tokenizer,
    hotpotqa,
    limit=CONFIG["hotpotqa_limit"]
)
hotpotqa_time = time.time() - hotpotqa_start

print(f"\n{'='*40}")
print(f"HOTPOTQA RESULTS")
print(f"{'='*40}")
print(f"F1 Score: {hotpotqa_results['f1']:.2%}")
print(f"Exact Match: {hotpotqa_results['em']:.2%}")
print(f"Total samples: {hotpotqa_results['total']}")
print(f"Time: {hotpotqa_time/60:.1f} minutes")

# Save results
hotpotqa_output = {
    'benchmark': 'hotpotqa',
    'model': CONFIG['model_name'],
    'f1': hotpotqa_results['f1'],
    'em': hotpotqa_results['em'],
    'total': hotpotqa_results['total'],
    'runtime_seconds': hotpotqa_time,
    'timestamp': CONFIG['timestamp']
}

with open(f"{CONFIG['results_dir']}/hotpotqa_results.json", 'w') as f:
    json.dump(hotpotqa_output, f, indent=2)
print(f"\n✅ Results saved to {CONFIG['results_dir']}/hotpotqa_results.json")

---

## 6. Benchmark 4: MMLU (General Knowledge)

**Description**: Massive Multitask Language Understanding - tests broad knowledge across 57 subjects.

**Metric**: Multiple-choice accuracy.

**Expected Baseline**: ~55-65% with 5-shot prompting

In [ ]:
# Cell 20: Load MMLU dataset
print("Loading MMLU dataset...")
mmlu = load_dataset("cais/mmlu", "all")
print(f"Test samples: {len(mmlu['test'])}")
print(f"Validation samples: {len(mmlu['validation'])}")
print(f"Dev samples: {len(mmlu['dev'])}")

# Show subjects
subjects = set(mmlu['test']['subject'])
print(f"\nNumber of subjects: {len(subjects)}")
print(f"Sample subjects: {list(subjects)[:5]}")

# Show example
example = mmlu['test'][0]
print("\n--- Example ---")
print(f"Subject: {example['subject']}")
print(f"Question: {example['question']}")
print(f"Choices: {example['choices']}")
print(f"Answer: {example['answer']} ({['A', 'B', 'C', 'D'][example['answer']]})")

In [ ]:
# Cell 21: MMLU evaluation functions
def create_mmlu_prompt(question, choices, fewshot_examples=None, subject=None):
    """Create few-shot prompt for MMLU"""

    prompt = "The following are multiple choice questions (with answers).\n\n"

    if fewshot_examples:
        for ex in fewshot_examples:
            prompt += f"{ex['question']}\n"
            for i, choice in enumerate(ex['choices']):
                prompt += f"{['A', 'B', 'C', 'D'][i]}. {choice}\n"
            prompt += f"Answer: {['A', 'B', 'C', 'D'][ex['answer']]}\n\n"

    prompt += f"{question}\n"
    for i, choice in enumerate(choices):
        prompt += f"{['A', 'B', 'C', 'D'][i]}. {choice}\n"
    prompt += "Answer:"

    return prompt

def parse_mmlu_response(response):
    """Extract answer letter from model response"""
    response = response.strip().upper()

    # Look for first A, B, C, or D
    for char in response:
        if char in ['A', 'B', 'C', 'D']:
            return ['A', 'B', 'C', 'D'].index(char)

    return -1  # No valid answer found

# Test parsing
test_responses = ["A", "The answer is B", "C. Paris", "I think D is correct"]
for resp in test_responses:
    print(f"Parse '{resp}' -> {parse_mmlu_response(resp)}")

In [ ]:
# Cell 22: Run MMLU evaluation
def evaluate_mmlu(model, tokenizer, dataset, num_fewshot=5, limit=None):
    """Evaluate model on MMLU benchmark"""

    test_data = dataset['test']
    dev_data = dataset['dev']  # For few-shot examples

    if limit:
        # Sample evenly across subjects
        subjects = list(set(test_data['subject']))
        samples_per_subject = max(1, limit // len(subjects))
        indices = []
        for subject in subjects:
            subject_indices = [i for i, s in enumerate(test_data['subject']) if s == subject]
            indices.extend(subject_indices[:samples_per_subject])
        test_data = test_data.select(indices[:limit])

    # Create subject-specific few-shot examples
    fewshot_by_subject = {}
    for subject in set(test_data['subject']):
        subject_dev = [ex for ex in dev_data if ex['subject'] == subject]
        fewshot_by_subject[subject] = subject_dev[:num_fewshot]

    results = []
    correct = 0
    total = 0
    subject_results = {}

    print(f"\nEvaluating MMLU ({len(test_data)} samples)...")

    for item in tqdm(test_data, desc="MMLU"):
        subject = item['subject']
        fewshot = fewshot_by_subject.get(subject, [])[:num_fewshot]

        prompt = create_mmlu_prompt(
            item['question'],
            item['choices'],
            fewshot_examples=fewshot,
            subject=subject
        )

        inputs = tokenizer(
            prompt,
            return_tensors="pt",
            truncation=True,
            max_length=2048
        ).to(model.device)

        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=5,
                do_sample=False,
                pad_token_id=tokenizer.eos_token_id
            )

        response = tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
        predicted = parse_mmlu_response(response)
        expected = item['answer']

        is_correct = predicted == expected
        if is_correct:
            correct += 1
        total += 1

        # Track per-subject results
        if subject not in subject_results:
            subject_results[subject] = {'correct': 0, 'total': 0}
        subject_results[subject]['total'] += 1
        if is_correct:
            subject_results[subject]['correct'] += 1

        results.append({
            'subject': subject,
            'question': item['question'][:200],
            'expected': ['A', 'B', 'C', 'D'][expected],
            'predicted': ['A', 'B', 'C', 'D'][predicted] if predicted >= 0 else 'NONE',
            'correct': is_correct
        })

        if total % 100 == 0:
            torch.cuda.empty_cache()

    # Calculate per-subject accuracies
    for subject in subject_results:
        subject_results[subject]['accuracy'] = (
            subject_results[subject]['correct'] / subject_results[subject]['total']
        )

    return {
        'accuracy': correct / total,
        'correct': correct,
        'total': total,
        'subject_results': subject_results,
        'results': results
    }

print("MMLU evaluation function ready.")

In [ ]:
# Cell 23: Execute MMLU evaluation
print("="*60)
print("BENCHMARK 4: MMLU (GENERAL KNOWLEDGE)")
print("="*60)

mmlu_start = time.time()
mmlu_results = evaluate_mmlu(
    model,
    tokenizer,
    mmlu,
    num_fewshot=CONFIG["mmlu_num_fewshot"],
    limit=CONFIG["mmlu_limit"]
)
mmlu_time = time.time() - mmlu_start

print(f"\n{'='*40}")
print(f"MMLU RESULTS")
print(f"{'='*40}")
print(f"Overall Accuracy: {mmlu_results['accuracy']:.2%}")
print(f"Correct: {mmlu_results['correct']}/{mmlu_results['total']}")
print(f"Time: {mmlu_time/60:.1f} minutes")

# Show per-category breakdown
print(f"\n--- Top 5 Subjects ---")
sorted_subjects = sorted(
    mmlu_results['subject_results'].items(),
    key=lambda x: x[1]['accuracy'],
    reverse=True
)[:5]
for subject, stats in sorted_subjects:
    print(f"  {subject}: {stats['accuracy']:.2%} ({stats['correct']}/{stats['total']})")

print(f"\n--- Bottom 5 Subjects ---")
for subject, stats in sorted_subjects[-5:][::-1]:
    print(f"  {subject}: {stats['accuracy']:.2%} ({stats['correct']}/{stats['total']})")

# Save results
mmlu_output = {
    'benchmark': 'mmlu',
    'model': CONFIG['model_name'],
    'accuracy': mmlu_results['accuracy'],
    'correct': mmlu_results['correct'],
    'total': mmlu_results['total'],
    'num_fewshot': CONFIG['mmlu_num_fewshot'],
    'subject_results': {
        k: {'accuracy': v['accuracy'], 'correct': v['correct'], 'total': v['total']}
        for k, v in mmlu_results['subject_results'].items()
    },
    'runtime_seconds': mmlu_time,
    'timestamp': CONFIG['timestamp']
}

with open(f"{CONFIG['results_dir']}/mmlu_results.json", 'w') as f:
    json.dump(mmlu_output, f, indent=2)
print(f"\n✅ Results saved to {CONFIG['results_dir']}/mmlu_results.json")

---

## 7. Results Summary

In [ ]:
# Cell 24: Compile all results
print("="*70)
print("SIGMA BASELINE BENCHMARKS: FINAL SUMMARY")
print("="*70)
print(f"\nModel: {CONFIG['model_name']}")
print(f"Timestamp: {CONFIG['timestamp']}")

# Create summary table
summary_data = {
    'Benchmark': ['GSM8K', 'ToolBench', 'HotpotQA', 'MMLU'],
    'Task Type': ['Reasoning', 'Tool-use', 'Retrieval', 'General'],
    'Metric': ['Accuracy', 'Pass Rate', 'F1 Score', 'Accuracy'],
    'Result': [
        f"{gsm8k_results['accuracy']:.2%}",
        f"{toolbench_results['full_accuracy']:.2%}",
        f"{hotpotqa_results['f1']:.2%}",
        f"{mmlu_results['accuracy']:.2%}"
    ],
    'Expected': ['45-55%', '40-50%', '35-45%', '55-65%'],
    'Samples': [
        gsm8k_results['total'],
        toolbench_results['total'],
        hotpotqa_results['total'],
        mmlu_results['total']
    ]
}

summary_df = pd.DataFrame(summary_data)
print("\n" + summary_df.to_string(index=False))

# Calculate total runtime
total_runtime = gsm8k_time + toolbench_time + hotpotqa_time + mmlu_time
print(f"\n{'='*40}")
print(f"Total Runtime: {total_runtime/3600:.2f} hours")
print(f"GPU Memory Peak: {torch.cuda.max_memory_allocated() / 1e9:.2f} GB")

In [ ]:
# Cell 25: Visualize results
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar chart of results
benchmarks = ['GSM8K\n(Reasoning)', 'ToolBench\n(Tool-use)', 'HotpotQA\n(Retrieval)', 'MMLU\n(General)']
results = [
    gsm8k_results['accuracy'] * 100,
    toolbench_results['full_accuracy'] * 100,
    hotpotqa_results['f1'] * 100,
    mmlu_results['accuracy'] * 100
]
expected_low = [45, 40, 35, 55]
expected_high = [55, 50, 45, 65]

colors = ['#2ecc71' if expected_low[i] <= results[i] <= expected_high[i] * 1.2 else '#e74c3c'
          for i in range(len(results))]

bars = axes[0].bar(benchmarks, results, color=colors, edgecolor='black', linewidth=1.5)

# Add expected range indicators
for i, (low, high) in enumerate(zip(expected_low, expected_high)):
    axes[0].hlines(y=low, xmin=i-0.4, xmax=i+0.4, colors='gray', linestyles='--', alpha=0.7)
    axes[0].hlines(y=high, xmin=i-0.4, xmax=i+0.4, colors='gray', linestyles='--', alpha=0.7)

axes[0].set_ylabel('Score (%)', fontsize=12)
axes[0].set_title(f'Jamba-1.5-Mini Baseline Results', fontsize=14, fontweight='bold')
axes[0].set_ylim(0, 100)

# Add value labels
for bar, val in zip(bars, results):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
                f'{val:.1f}%', ha='center', va='bottom', fontsize=11, fontweight='bold')

# Runtime breakdown pie chart
runtimes = [gsm8k_time, toolbench_time, hotpotqa_time, mmlu_time]
runtime_labels = ['GSM8K', 'ToolBench', 'HotpotQA', 'MMLU']
colors_pie = ['#3498db', '#e74c3c', '#2ecc71', '#f39c12']

axes[1].pie(runtimes, labels=runtime_labels, autopct='%1.1f%%', colors=colors_pie,
            explode=[0.02]*4, startangle=90)
axes[1].set_title(f'Runtime Distribution\n(Total: {total_runtime/60:.0f} min)', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.savefig(f"{CONFIG['results_dir']}/baseline_results_summary.png", dpi=150, bbox_inches='tight')
plt.show()

print(f"\n✅ Visualization saved to {CONFIG['results_dir']}/baseline_results_summary.png")

---

## 8. Export Results

In [ ]:
# Cell 26: Export comprehensive results
# Compile all results into a single JSON file
comprehensive_results = {
    'experiment_info': {
        'project': 'SIGMA',
        'phase': 'Phase 0 - Baseline Establishment',
        'model': CONFIG['model_name'],
        'timestamp': CONFIG['timestamp'],
        'total_runtime_hours': total_runtime / 3600,
        'gpu': torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'N/A',
        'gpu_memory_peak_gb': torch.cuda.max_memory_allocated() / 1e9
    },
    'benchmarks': {
        'gsm8k': {
            'task_type': 'reasoning',
            'metric': 'accuracy',
            'score': gsm8k_results['accuracy'],
            'correct': gsm8k_results['correct'],
            'total': gsm8k_results['total'],
            'num_fewshot': CONFIG['gsm8k_num_fewshot'],
            'runtime_minutes': gsm8k_time / 60
        },
        'toolbench': {
            'task_type': 'tool-use',
            'metric': 'pass_rate',
            'score': toolbench_results['full_accuracy'],
            'tool_accuracy': toolbench_results['tool_accuracy'],
            'correct': toolbench_results['full_correct'],
            'total': toolbench_results['total'],
            'runtime_minutes': toolbench_time / 60
        },
        'hotpotqa': {
            'task_type': 'retrieval',
            'metric': 'f1',
            'score': hotpotqa_results['f1'],
            'em': hotpotqa_results['em'],
            'total': hotpotqa_results['total'],
            'runtime_minutes': hotpotqa_time / 60
        },
        'mmlu': {
            'task_type': 'general',
            'metric': 'accuracy',
            'score': mmlu_results['accuracy'],
            'correct': mmlu_results['correct'],
            'total': mmlu_results['total'],
            'num_fewshot': CONFIG['mmlu_num_fewshot'],
            'runtime_minutes': mmlu_time / 60
        }
    },
    'config': CONFIG
}

# Save comprehensive results
with open(f"{CONFIG['results_dir']}/comprehensive_baseline_results.json", 'w') as f:
    json.dump(comprehensive_results, f, indent=2)

print(f"✅ Comprehensive results saved to {CONFIG['results_dir']}/comprehensive_baseline_results.json")

In [ ]:
# Cell 27: Generate markdown report
report = f"""# SIGMA Phase 0: Jamba-1.5-Mini Baseline Results

**Date**: {datetime.now().strftime('%Y-%m-%d %H:%M')}
**Model**: {CONFIG['model_name']}
**Total Runtime**: {total_runtime/3600:.2f} hours

---

## Summary Table

| Benchmark | Task Type | Metric | Score | Expected | Status |
|-----------|-----------|--------|-------|----------|--------|
| GSM8K | Reasoning | Accuracy | {gsm8k_results['accuracy']:.2%} | 45-55% | {'✅' if 0.45 <= gsm8k_results['accuracy'] <= 0.55 else '⚠️'} |
| ToolBench | Tool-use | Pass Rate | {toolbench_results['full_accuracy']:.2%} | 40-50% | {'✅' if 0.40 <= toolbench_results['full_accuracy'] <= 0.50 else '⚠️'} |
| HotpotQA | Retrieval | F1 | {hotpotqa_results['f1']:.2%} | 35-45% | {'✅' if 0.35 <= hotpotqa_results['f1'] <= 0.45 else '⚠️'} |
| MMLU | General | Accuracy | {mmlu_results['accuracy']:.2%} | 55-65% | {'✅' if 0.55 <= mmlu_results['accuracy'] <= 0.65 else '⚠️'} |

---

## Detailed Results

### GSM8K (Reasoning)
- **Accuracy**: {gsm8k_results['accuracy']:.2%} ({gsm8k_results['correct']}/{gsm8k_results['total']})
- **Few-shot**: {CONFIG['gsm8k_num_fewshot']}-shot
- **Runtime**: {gsm8k_time/60:.1f} minutes

### ToolBench (Tool-use)
- **Pass Rate**: {toolbench_results['full_accuracy']:.2%} ({toolbench_results['full_correct']}/{toolbench_results['total']})
- **Tool Selection**: {toolbench_results['tool_accuracy']:.2%}
- **Runtime**: {toolbench_time/60:.1f} minutes

### HotpotQA (Retrieval)
- **F1 Score**: {hotpotqa_results['f1']:.2%}
- **Exact Match**: {hotpotqa_results['em']:.2%}
- **Samples**: {hotpotqa_results['total']}
- **Runtime**: {hotpotqa_time/60:.1f} minutes

### MMLU (General)
- **Accuracy**: {mmlu_results['accuracy']:.2%} ({mmlu_results['correct']}/{mmlu_results['total']})
- **Few-shot**: {CONFIG['mmlu_num_fewshot']}-shot
- **Runtime**: {mmlu_time/60:.1f} minutes

---

## Next Steps

These baseline results establish the **Baseline A** (Vanilla Jamba-1.5-Mini) for SIGMA evaluation.

1. ✅ Phase 0 Complete: Baseline established
2. ⏳ Phase 1: Implement TaskAwareRouter
3. ⏳ Phase 2: Train task-specialized LoRA adapters
4. ⏳ Phase 3: Compare SIGMA vs. Baseline

---

*Generated by SIGMA Baseline Benchmark Notebook*
"""

with open(f"{CONFIG['results_dir']}/baseline_report.md", 'w') as f:
    f.write(report)

print(f"✅ Report saved to {CONFIG['results_dir']}/baseline_report.md")
print("\n" + report)

In [ ]:
# Cell 28: Final cleanup and summary
print("="*70)
print("BENCHMARK EVALUATION COMPLETE")
print("="*70)
print(f"\n📁 All results saved to: {CONFIG['results_dir']}/")
print("\n📄 Files generated:")
print("   - gsm8k_results.json")
print("   - toolbench_results.json")
print("   - hotpotqa_results.json")
print("   - mmlu_results.json")
print("   - comprehensive_baseline_results.json")
print("   - baseline_results_summary.png")
print("   - baseline_report.md")
print("\n🎯 Next: Use these baselines to evaluate SIGMA's task-aware routing!")

# Clear GPU memory
del model
torch.cuda.empty_cache()
print(f"\n🧹 GPU memory cleared")

---

## Appendix: Using lm-evaluation-harness (Alternative)

For standardized evaluation, you can also use the lm-evaluation-harness command line tool:

In [ ]:
# Cell A1: Alternative - Run GSM8K via lm-eval CLI
# Uncomment to use command-line evaluation

# !lm_eval --model hf \
#     --model_args pretrained=ai21labs/AI21-Jamba-1.5-Mini,dtype=float16,trust_remote_code=True \
#     --tasks gsm8k \
#     --num_fewshot 8 \
#     --batch_size 1 \
#     --output_path ./lm_eval_results/gsm8k

In [ ]:
# Cell A2: Alternative - Run MMLU via lm-eval CLI
# Uncomment to use command-line evaluation

# !lm_eval --model hf \
#     --model_args pretrained=ai21labs/AI21-Jamba-1.5-Mini,dtype=float16,trust_remote_code=True \
#     --tasks mmlu \
#     --num_fewshot 5 \
#     --batch_size 1 \
#     --output_path ./lm_eval_results/mmlu

---

## Quick Reference: Expected vs Actual

Update this table with your actual results:

| Benchmark | Expected | Actual | Delta |
|-----------|----------|--------|-------|
| GSM8K | 45-55% | __%  | __ |
| ToolBench | 40-50% | __% | __ |
| HotpotQA | 35-45% | __% | __ |
| MMLU | 55-65% | __% | __ |

---

*Notebook created for SIGMA Project - AAI/CPE/EE 800 Spring 2026*

# Task
Update the `CONFIG` dictionary to enable 4-bit quantization, clear the GPU memory, and then modify 'Cell 6' to load the model using `BitsAndBytesConfig` with 4-bit quantization and `torch.float16` compute dtype to resolve the `RuntimeError` due to memory constraints. Subsequently, re-run 'Cell 7' to verify the model loading and then re-run 'Cell 11' to restart the GSM8K evaluation. Upon successful completion of GSM8K, proceed to evaluate the model on ToolBench, HotpotQA, and MMLU by running 'Cell 15', 'Cell 19', and 'Cell 23' respectively. Finally, summarize, visualize, and export all benchmark results by running 'Cell 24', 'Cell 25', 'Cell 26', and 'Cell 27'.

## Update Configuration for Quantization

### Subtask:
Modify the `CONFIG` dictionary to include `load_in_4bit=True` and `bnb_4bit_compute_dtype=torch.float16` settings for 4-bit quantization, which will reduce the model's memory footprint significantly. This will be added to the CONFIG dictionary before model loading.


**Reasoning**:
To implement 4-bit quantization, I need to modify the `CONFIG` dictionary by adding `quantization_config` and setting `torch_dtype` to `None`. This requires importing `BitsAndBytesConfig` from `transformers`.



In [ ]:
from transformers import BitsAndBytesConfig

print("Updating CONFIG for 4-bit quantization...")

# Configure 4-bit quantization
CONFIG['quantization_config'] = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16
)

# Set torch_dtype to None to avoid conflicts when using quantization
CONFIG['torch_dtype'] = None

print("✅ CONFIG updated with quantization settings.")
print(f"load_in_4bit: {CONFIG['quantization_config'].load_in_4bit}")
print(f"bnb_4bit_compute_dtype: {CONFIG['quantization_config'].bnb_4bit_compute_dtype}")
print(f"torch_dtype: {CONFIG['torch_dtype']}")

**Reasoning**:
The previous model loading failed due to memory constraints, and the previous error logs indicate `RuntimeError: Tensor on device meta is not on the expected device cuda:0!`. To resolve this, I need to clear the GPU memory first and then modify 'Cell 6' to correctly load the model using the newly configured `quantization_config` for 4-bit quantization. This will reduce memory usage and prevent the `RuntimeError`.



In [ ]:
# Cell 6: Load Jamba-1.5-Mini model and tokenizer
from transformers import AutoModelForCausalLM, AutoTokenizer, JambaConfig

# Clear existing model and tokenizer from memory if they exist
if 'model' in locals():
    del model
if 'tokenizer' in locals():
    del tokenizer

# Clear GPU cache after deleting the model
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    print("🧹 GPU memory cleared before loading new model.")

print(f"Loading {CONFIG['model_name']} with 4-bit quantization...")
print("This may take 5-10 minutes for first download (~24GB)")

start_time = time.time()

tokenizer = AutoTokenizer.from_pretrained(
    CONFIG["model_name"],
    trust_remote_code=True
)

# Ensure padding token is set
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# CRITICAL: Load config and disable fast Mamba kernels to avoid CUDA mismatch
print("Loading model configuration...")
config = JambaConfig.from_pretrained(CONFIG["model_name"])
config.use_mamba_kernels = False  # Disable fast kernels for CUDA compatibility
print("⚙️ Mamba kernels: DISABLED (for compatibility)")

model = AutoModelForCausalLM.from_pretrained(
    CONFIG["model_name"],
    config=config,  # Use modified config
    quantization_config=CONFIG['quantization_config'], # Apply 4-bit quantization
    device_map=CONFIG["device_map"],
    trust_remote_code=True,
    # attn_implementation="flash_attention_2"  # Commented out - will use default
)

model.eval()

load_time = time.time() - start_time
print(f"\n✅ Model loaded in {load_time:.1f} seconds")
print(f"Parameters: {sum(p.numel() for p in model.parameters()) / 1e9:.2f}B total")
if torch.cuda.is_available():
    print(f"GPU Memory: {torch.cuda.memory_allocated() / 1e9:.2f} GB allocated")
print(f"Kernels: Fallback mode (compatible, ~5-10% slower)")

**Reasoning**:
The previous error indicated that some modules were being dispatched to CPU or disk without `llm_int8_enable_fp32_cpu_offload=True` being set in the quantization configuration. To resolve this, I need to add `llm_int8_enable_fp32_cpu_offload=True` to the `BitsAndBytesConfig` within the `CONFIG` dictionary.



In [ ]:
from transformers import BitsAndBytesConfig
import torch

print("Updating CONFIG for 4-bit quantization with CPU offload compatibility...")

# Configure 4-bit quantization, enabling CPU offload for 32-bit modules if necessary
CONFIG['quantization_config'] = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    llm_int8_enable_fp32_cpu_offload=True # Added to resolve the ValueError
)

# Set torch_dtype to None to avoid conflicts when using quantization
CONFIG['torch_dtype'] = None

print("✅ CONFIG updated with quantization settings.")
print(f"load_in_4bit: {CONFIG['quantization_config'].load_in_4bit}")
print(f"bnb_4bit_compute_dtype: {CONFIG['quantization_config'].bnb_4bit_compute_dtype}")
print(f"llm_int8_enable_fp32_cpu_offload: {CONFIG['quantization_config'].llm_int8_enable_fp32_cpu_offload}")
print(f"torch_dtype: {CONFIG['torch_dtype']}")

**Reasoning**:
With the `CONFIG` updated to correctly handle CPU offloading for 4-bit quantization, the next step is to re-run 'Cell 6' to load the model using the updated `quantization_config`. This should resolve the previous `ValueError` and successfully load the Jamba-1.5-Mini model.



In [ ]:
# Cell 6: Load Jamba-1.5-Mini model and tokenizer
from transformers import AutoModelForCausalLM, AutoTokenizer, JambaConfig

# Clear existing model and tokenizer from memory if they exist
if 'model' in locals():
    del model
if 'tokenizer' in locals():
    del tokenizer

# Clear GPU cache after deleting the model
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    print("🧹 GPU memory cleared before loading new model.")

print(f"Loading {CONFIG['model_name']} with 4-bit quantization...")
print("This may take 5-10 minutes for first download (~24GB)")

start_time = time.time()

tokenizer = AutoTokenizer.from_pretrained(
    CONFIG["model_name"],
    trust_remote_code=True
)

# Ensure padding token is set
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# CRITICAL: Load config and disable fast Mamba kernels to avoid CUDA mismatch
print("Loading model configuration...")
config = JambaConfig.from_pretrained(CONFIG["model_name"])
config.use_mamba_kernels = False  # Disable fast kernels for CUDA compatibility
print("⚙️ Mamba kernels: DISABLED (for compatibility)")

model = AutoModelForCausalLM.from_pretrained(
    CONFIG["model_name"],
    config=config,  # Use modified config
    quantization_config=CONFIG['quantization_config'], # Apply 4-bit quantization
    device_map=CONFIG["device_map"],
    trust_remote_code=True,
    # attn_implementation="flash_attention_2"  # Commented out - will use default
)

model.eval()

load_time = time.time() - start_time
print(f"\n✅ Model loaded in {load_time:.1f} seconds")
print(f"Parameters: {sum(p.numel() for p in model.parameters()) / 1e9:.2f}B total")
if torch.cuda.is_available():
    print(f"GPU Memory: {torch.cuda.memory_allocated() / 1e9:.2f} GB allocated")
print(f"Kernels: Fallback mode (compatible, ~5-10% slower)")

**Reasoning**:
The previous error, `TypeError: Params4bit.__new__() got an unexpected keyword argument '_is_hf_initialized'`, suggests an incompatibility in how the `accelerate` library passes arguments to `bitsandbytes` during model loading, potentially due to version mismatch or specific model configuration. To address this within the constraints of modifying 'Cell 6', I will revert to using `AutoConfig.from_pretrained` (as in the original notebook) and re-introduce `low_cpu_mem_usage=True` and `offload_buffers=True` to influence how `accelerate` handles memory and dispatches the model. These parameters can sometimes change the execution path and resolve such type errors, by allowing `accelerate` more flexibility in device placement, which might bypass the problematic `Params4bit` initialization.



In [ ]:
# Cell 6: Load Jamba-1.5-Mini model and tokenizer
from transformers import AutoModelForCausalLM, AutoTokenizer, AutoConfig # Changed from JambaConfig

# Clear existing model and tokenizer from memory if they exist
if 'model' in locals():
    del model
if 'tokenizer' in locals():
    del tokenizer

# Clear GPU cache after deleting the model
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    print("🧹 GPU memory cleared before loading new model.")

print(f"Loading {CONFIG['model_name']} with 4-bit quantization...")
print("This may take 5-10 minutes for first download (~24GB)")

start_time = time.time()

tokenizer = AutoTokenizer.from_pretrained(
    CONFIG["model_name"],
    trust_remote_code=True
)

# Ensure padding token is set
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# CRITICAL: Load config and disable fast Mamba kernels to avoid CUDA mismatch
print("Loading model configuration...")
config = AutoConfig.from_pretrained( # Changed from JambaConfig
    CONFIG["model_name"],
    trust_remote_code=True
)
if hasattr(config, 'use_mamba_kernels'): # Re-added conditional check for mamba kernels
    config.use_mamba_kernels = False
    print("⚙️ Mamba kernels: DISABLED (for compatibility)")

model = AutoModelForCausalLM.from_pretrained(
    CONFIG["model_name"],
    config=config,  # Use modified config
    quantization_config=CONFIG['quantization_config'], # Apply 4-bit quantization
    device_map=CONFIG["device_map"],
    trust_remote_code=True,
    low_cpu_mem_usage=True, # Re-added to help with memory dispatching
    offload_buffers=True,   # Re-added to help with memory dispatching
    # attn_implementation="flash_attention_2"  # Commented out - will use default
)

model.eval()

load_time = time.time() - start_time
print(f"\n✅ Model loaded in {load_time:.1f} seconds")
print(f"Parameters: {sum(p.numel() for p in model.parameters()) / 1e9:.2f}B total")
if torch.cuda.is_available():
    print(f"GPU Memory: {torch.cuda.memory_allocated() / 1e9:.2f} GB allocated")
print(f"Kernels: Fallback mode (compatible, ~5-10% slower)")